# 🌲 Tree-Based Models for Tabular Data
### Decision Trees · Random Forests · Gradient Boosting · LightGBM · CatBoost · XGBoost

This notebook walks through **all nine tasks** of the assignment step by step.  
Each section is self-contained and ends with a result/interpretation cell.

---
**Table of Contents**
1. [Setup & Imports](#1)
2. [Data Download & Preprocessing](#2)
3. [Custom Decision Tree (Classifier & Regressor)](#3)
4. [Performance of Custom DT vs sklearn DT](#4)
5. [Random Forest Classifier](#5)
6. [Custom GBDT Classifier](#6)
7. [LightGBM · CatBoost · XGBoost](#7)
8. [Best Model — Test Evaluation](#8)
9. [ExtraTrees Classifier *(Bonus)*](#9)


## 1 · Setup & Imports <a id='1'></a>
Install any missing libraries and import everything we need throughout the notebook.

In [1]:
# Install third-party GBDT libraries (run once)
# !pip install lightgbm catboost xgboost category_encoders joblib --quiet


In [2]:
import warnings, os, random, multiprocessing
import numpy as np
import pandas as pd
from collections import Counter
from joblib import Parallel, delayed, cpu_count

# ── sklearn utilities ──────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier as SklearnDTC
from sklearn.ensemble import RandomForestClassifier as SklearnRFC
from sklearn.metrics import roc_auc_score

# ── GBDT libraries ─────────────────────────────────────────────────────────
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

warnings.filterwarnings("ignore")
SEED    = 42
N_JOBS  = -1          # use ALL logical cores everywhere
N_CORES = cpu_count()
random.seed(SEED)
np.random.seed(SEED)

def gini(y_true, y_score):
    """Normalised Gini = 2*AUC - 1  (0..1, higher is better)."""
    return 2 * roc_auc_score(y_true, y_score) - 1

print(f"All imports OK ✓  |  logical cores available: {N_CORES}")


All imports OK ✓  |  logical cores available: 32


---
## 2 · Data Download & Preprocessing <a id='2'></a>

### Dataset: *Don't Get Kicked!* (Kaggle)
Predict whether a car purchased at auction turns out to be a bad buy (*IsBadBuy* = 1).

**Split strategy (time-based)**  
We sort by `PurchDate` and cut the sorted data into three equal thirds:
- **Train** — first 33 %  
- **Valid** — middle 33 %  
- **Test** — last 33 % ← *do not touch until Task 8!*

**Why time-based?**  
Random splits would leak future information into the training set, causing overly optimistic validation scores that don't reflect real deployment.

**Encoding**  
Categorical columns are label-encoded. The encoder is **fit only on the training set** and then applied to validation and test to prevent data leakage.  
Unseen categories in valid/test are mapped to a dedicated `"__unknown__"` bucket before encoding.


In [3]:
# ── 2.1  Load data ────────────────────────────────────────────────────────
os.makedirs("data", exist_ok=True)

# Place training.csv inside ./data/  OR run:
# !kaggle competitions download -c DontGetKicked -p data/ --unzip

CSV_PATH = "data/training.csv"
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Shape: {df.shape}")
df.head(3)


Shape: (72983, 34)


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389


In [4]:
# ── 2.2  Time-based split ─────────────────────────────────────────────────
df["PurchDate"] = pd.to_datetime(df["PurchDate"], errors="coerce")
df = df.sort_values("PurchDate").reset_index(drop=True)

n = len(df)
t1, t2 = n // 3, 2 * n // 3

train_df = df.iloc[:t1].copy()
valid_df = df.iloc[t1:t2].copy()
test_df  = df.iloc[t2:].copy()

print(f"Train : {len(train_df):>6}  rows  |  {train_df.PurchDate.min().date()} → {train_df.PurchDate.max().date()}")
print(f"Valid : {len(valid_df):>6}  rows  |  {valid_df.PurchDate.min().date()} → {valid_df.PurchDate.max().date()}")
print(f"Test  : {len(test_df):>6}  rows  |  {test_df.PurchDate.min().date()} → {test_df.PurchDate.max().date()}")


Train :  24327  rows  |  2009-01-05 → 2009-09-15
Valid :  24328  rows  |  2009-09-15 → 2010-05-14
Test  :  24328  rows  |  2010-05-14 → 2010-12-30


In [5]:
# ── 2.3  Feature engineering & encoding ──────────────────────────────────
TARGET    = "IsBadBuy"
DROP_COLS = ["RefId", "PurchDate", "WheelTypeID", "BYRNO", "VNZIP1", "VNST"]

feature_cols = [c for c in df.columns if c not in DROP_COLS + [TARGET]]
cat_cols     = [c for c in feature_cols if train_df[c].dtype == "object"]
num_cols     = [c for c in feature_cols if c not in cat_cols]

# Fill NaN
for split in [train_df, valid_df, test_df]:
    split[num_cols] = split[num_cols].fillna(train_df[num_cols].median())
    split[cat_cols] = split[cat_cols].fillna("__missing__")

# Label-encode cats (fit on train only; unseen → "__unknown__")
encoders = {}
for col in cat_cols:
    known = set(train_df[col].unique())
    for split in [train_df, valid_df, test_df]:
        split[col] = split[col].apply(lambda x, k=known: x if x in k else "__unknown__")
    le = LabelEncoder()
    le.fit(list(known) + ["__unknown__"])
    for split in [train_df, valid_df, test_df]:
        split[col] = le.transform(split[col])
    encoders[col] = le

# Numpy arrays for custom models / LightGBM / XGBoost
X_train = train_df[feature_cols].values.astype(float)
y_train = train_df[TARGET].values.astype(int)
X_valid = valid_df[feature_cols].values.astype(float)
y_valid = valid_df[TARGET].values.astype(int)
X_test  = test_df[feature_cols].values.astype(float)
y_test  = test_df[TARGET].values.astype(int)

# DataFrames with original strings for CatBoost
def make_cb_df(split_df):
    out = {}
    for col in feature_cols:
        if col in cat_cols:
            out[col] = encoders[col].inverse_transform(split_df[col].astype(int).values)
        else:
            out[col] = split_df[col].values
    return pd.DataFrame(out)

cb_train_df = make_cb_df(train_df)
cb_valid_df = make_cb_df(valid_df)
cb_test_df  = make_cb_df(test_df)

print(f"X_train: {X_train.shape}  |  positive rate: {y_train.mean():.3f}")
print(f"X_valid: {X_valid.shape}  |  positive rate: {y_valid.mean():.3f}")
print(f"X_test : {X_test.shape}   |  positive rate: {y_test.mean():.3f}")
print(f"\nFeatures ({len(feature_cols)}): {feature_cols[:8]} ...")


X_train: (24327, 27)  |  positive rate: 0.115
X_valid: (24328, 27)  |  positive rate: 0.130
X_test : (24328, 27)   |  positive rate: 0.124

Features (27): ['Auction', 'VehYear', 'VehicleAge', 'Make', 'Model', 'Trim', 'SubModel', 'Color'] ...


---
## 3 · Custom Decision Tree (Classifier & Regressor) <a id='3'></a>

### Theory recap
A decision tree recursively partitions the feature space with **axis-aligned splits**.  
At each node we choose the feature and threshold that maximally reduces impurity.

| Model | Impurity criterion |
|---|---|
| Classifier | **Gini impurity** = 1 − Σ pᵢ² |
| Regressor  | **Variance** (std deviation) |

**Gini impurity** measures how often a randomly chosen element from the set would be  
incorrectly labelled if it was randomly labelled according to the class distribution.  
A Gini of 0 means the node is perfectly pure.

**Greedy tree growing:**  
We pick the single best split at every node (no look-ahead).  
This is *not* globally optimal but is tractable — finding the true optimum is NP-complete.

### Node class
Each node stores:
- the feature index and threshold used to split
- left / right child pointers
- leaf prediction value (only set for leaf nodes)


In [6]:
# ══════════════════════════════════════════════════════════════════════════
#  Custom Decision Tree — full implementation
#
#  Speed improvements applied:
#    • Threshold candidates capped at 20 percentile points (avoids O(n²))
#    • _best_split uses vectorised numpy ops instead of Python loops
#    • predict_proba uses a tight Python stack-based traversal
# ══════════════════════════════════════════════════════════════════════════

class Node:
    """A single node in a decision tree."""
    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, *, value=None):
        self.feature   = feature
        self.threshold = threshold
        self.left      = left
        self.right     = right
        self.value     = value

    def is_leaf(self):
        return self.value is not None


def _gini(y):
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    p = counts / len(y)
    return 1.0 - float(np.dot(p, p))          # vectorised dot product


def _variance(y):
    return float(np.var(y)) if len(y) > 1 else 0.0


# ── top-level helper so joblib can pickle it ──────────────────────────────
def _eval_feature_clf(feat, X_col, y, n_samples, base_imp, max_thresh=20):
    """Evaluate all threshold candidates for one feature (classifier)."""
    uniq = np.unique(X_col)
    if len(uniq) > max_thresh:
        uniq = np.percentile(X_col, np.linspace(0, 100, max_thresh))
        uniq = np.unique(uniq)
    best_gain, best_thresh = -np.inf, None
    for thresh in uniq:
        lm = X_col <= thresh
        nl, nr = lm.sum(), (~lm).sum()
        if nl == 0 or nr == 0:
            continue
        gain = base_imp - (nl / n_samples * _gini(y[lm]) +
                           nr / n_samples * _gini(y[~lm]))
        if gain > best_gain:
            best_gain, best_thresh = gain, thresh
    return feat, best_thresh, best_gain


def _eval_feature_reg(feat, X_col, y, n_samples, base_var, max_thresh=20):
    """Evaluate all threshold candidates for one feature (regressor)."""
    uniq = np.unique(X_col)
    if len(uniq) > max_thresh:
        uniq = np.percentile(X_col, np.linspace(0, 100, max_thresh))
        uniq = np.unique(uniq)
    best_gain, best_thresh = -np.inf, None
    for thresh in uniq:
        lm = X_col <= thresh
        nl, nr = lm.sum(), (~lm).sum()
        if nl == 0 or nr == 0:
            continue
        gain = base_var - (nl / n_samples * _variance(y[lm]) +
                           nr / n_samples * _variance(y[~lm]))
        if gain > best_gain:
            best_gain, best_thresh = gain, thresh
    return feat, best_thresh, best_gain


# ══════════════════════════════════════════════════════════════════════════
#  Decision Tree Classifier
# ══════════════════════════════════════════════════════════════════════════
class DecisionTreeClassifier:
    """Gini-based decision tree classifier.

    Parameters
    ----------
    max_depth          : int | None
    min_samples_split  : int
    max_features       : int | None   — features sampled per split
    extra_randomized   : bool         — use random thresholds (ExtraTrees)
    n_jobs             : int          — parallel feature evaluation per node
    random_state       : int | None
    """
    def __init__(self, max_depth=None, min_samples_split=2,
                 max_features=None, extra_randomized=False,
                 n_jobs=1, random_state=None):
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.max_features      = max_features
        self.extra_randomized  = extra_randomized
        self.n_jobs            = n_jobs
        self.random_state      = random_state
        self.root              = None
        self._rng              = np.random.RandomState(random_state)

    # ── public API ──────────────────────────────────────────────────────────
    def fit(self, X, y):
        self.n_classes_ = int(y.max()) + 1
        self.root = self._grow(np.asarray(X, float), np.asarray(y, int), 0)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, float)
        out = np.empty((len(X), self.n_classes_))
        for i, x in enumerate(X):
            node = self.root
            while not node.is_leaf():
                node = node.left if x[node.feature] <= node.threshold else node.right
            out[i] = node.value
        return out

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    # ── tree growing ────────────────────────────────────────────────────────
    def _grow(self, X, y, depth):
        n, nf = X.shape
        if ((self.max_depth is not None and depth >= self.max_depth)
                or n < self.min_samples_split
                or len(np.unique(y)) == 1):
            return Node(value=self._leaf_proba(y))

        feat, thresh = (self._split_random(X, y)
                        if self.extra_randomized
                        else self._split_best(X, y))
        if feat is None:
            return Node(value=self._leaf_proba(y))

        lm = X[:, feat] <= thresh
        return Node(feature=feat, threshold=thresh,
                    left =self._grow(X[ lm], y[ lm], depth + 1),
                    right=self._grow(X[~lm], y[~lm], depth + 1))

    def _feat_idx(self, nf):
        if self.max_features is None or self.max_features >= nf:
            return np.arange(nf)
        return self._rng.choice(nf, self.max_features, replace=False)

    def _split_best(self, X, y):
        n, nf = X.shape
        base  = _gini(y)
        feats = self._feat_idx(nf)
        results = Parallel(n_jobs=self.n_jobs, prefer="threads")(
            delayed(_eval_feature_clf)(f, X[:, f], y, n, base)
            for f in feats
        )
        best = max(results, key=lambda r: r[2] if r[1] is not None else -np.inf)
        return (best[0], best[1]) if best[1] is not None else (None, None)

    def _split_random(self, X, y):
        n, nf = X.shape
        base  = _gini(y)
        feats = self._feat_idx(nf)
        best_gain, best_feat, best_thresh = -np.inf, None, None
        for f in feats:
            lo, hi = X[:, f].min(), X[:, f].max()
            if lo == hi:
                continue
            thresh = self._rng.uniform(lo, hi)
            lm = X[:, f] <= thresh
            nl, nr = lm.sum(), (~lm).sum()
            if nl == 0 or nr == 0:
                continue
            gain = base - (nl / n * _gini(y[lm]) + nr / n * _gini(y[~lm]))
            if gain > best_gain:
                best_gain, best_feat, best_thresh = gain, f, thresh
        return best_feat, best_thresh

    def _leaf_proba(self, y):
        counts = np.bincount(y, minlength=self.n_classes_)
        return counts / counts.sum()


# ══════════════════════════════════════════════════════════════════════════
#  Decision Tree Regressor  (variance / MSE criterion)
# ══════════════════════════════════════════════════════════════════════════
class DecisionTreeRegressor:
    """Variance-reduction (MSE) regression tree."""
    def __init__(self, max_depth=None, min_samples_split=2,
                 max_features=None, n_jobs=1, random_state=None):
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.max_features      = max_features
        self.n_jobs            = n_jobs
        self.root              = None
        self._rng              = np.random.RandomState(random_state)

    def fit(self, X, y):
        self.root = self._grow(np.asarray(X, float), np.asarray(y, float), 0)
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        out = np.empty(len(X))
        for i, x in enumerate(X):
            node = self.root
            while not node.is_leaf():
                node = node.left if x[node.feature] <= node.threshold else node.right
            out[i] = node.value
        return out

    def _grow(self, X, y, depth):
        n, nf = X.shape
        if ((self.max_depth is not None and depth >= self.max_depth)
                or n < self.min_samples_split
                or np.var(y) < 1e-10):
            return Node(value=float(np.mean(y)))

        nf_use = (self.max_features if self.max_features
                  else nf)
        feats  = self._rng.choice(nf, min(nf_use, nf), replace=False)
        base   = _variance(y)

        results = Parallel(n_jobs=self.n_jobs, prefer="threads")(
            delayed(_eval_feature_reg)(f, X[:, f], y, n, base)
            for f in feats
        )
        best = max(results, key=lambda r: r[2] if r[1] is not None else -np.inf)
        feat, thresh = best[0], best[1]
        if thresh is None:
            return Node(value=float(np.mean(y)))

        lm = X[:, feat] <= thresh
        return Node(feature=feat, threshold=thresh,
                    left =self._grow(X[ lm], y[ lm], depth + 1),
                    right=self._grow(X[~lm], y[~lm], depth + 1))


print("Custom Decision Tree classes defined ✓")
print(f"(threshold search parallelised across {N_CORES} cores per node)")


Custom Decision Tree classes defined ✓
(threshold search parallelised across 32 cores per node)


---
## 4 · Performance: Custom DT vs. sklearn DT <a id='4'></a>

We train our custom `DecisionTreeClassifier` and sklearn's implementation with the same  
`max_depth`, then compare Gini scores.

> **Target:** Gini ≥ 0.10 on the validation set (Task 3).

**Why might sklearn be better?**  
- sklearn uses optimised C-compiled code (faster and allows more splits to be evaluated).  
- sklearn's implementation handles ties and edge cases more carefully.  
- The exhaustive split search over all thresholds in sklearn is more thorough.  
- sklearn supports pruning methods (`min_impurity_decrease`, `ccp_alpha`).


In [7]:
# ── 4.1  Custom DT ───────────────────────────────────────────────────────
print("Training Custom Decision Tree  (max_depth=7)…")
custom_dt = DecisionTreeClassifier(max_depth=7, random_state=SEED, n_jobs=N_JOBS)
custom_dt.fit(X_train, y_train)

prob_custom = custom_dt.predict_proba(X_valid)[:, 1]
gini_custom = gini(y_valid, prob_custom)
print(f"  Custom DT  →  Gini (valid) = {gini_custom:.4f}")

# ── 4.2  sklearn DT (compiled C — benchmark) ─────────────────────────────
print("\nTraining sklearn Decision Tree (max_depth=7)…")
sk_dt = SklearnDTC(max_depth=7, random_state=SEED, criterion="gini")
sk_dt.fit(X_train, y_train)

prob_sk = sk_dt.predict_proba(X_valid)[:, 1]
gini_sk = gini(y_valid, prob_sk)
print(f"  sklearn DT →  Gini (valid) = {gini_sk:.4f}")

results = pd.DataFrame({
    "Model"       : ["Custom DT", "sklearn DT"],
    "Gini (valid)": [round(gini_custom, 4), round(gini_sk, 4)],
})
print("\n", results.to_string(index=False))
print(f"\n{'✓ Task 3 target met!' if gini_custom >= 0.10 else '✗ Gini < 0.10 — try increasing max_depth'}")


Training Custom Decision Tree  (max_depth=7)…
  Custom DT  →  Gini (valid) = 0.3927

Training sklearn Decision Tree (max_depth=7)…
  sklearn DT →  Gini (valid) = 0.3429

      Model  Gini (valid)
 Custom DT        0.3927
sklearn DT        0.3429

✓ Task 3 target met!


---
## 5 · Random Forest Classifier <a id='5'></a>

### How Random Forest works
Random Forest is a **bagging** ensemble: we train many deep decision trees on  
**random row + feature subsamples** of the training set, then average their  
probability predictions.

Key ideas:
- **Row subsampling** — each tree sees ~63 % of rows (sampling with replacement).
- **Feature subsampling** — at every split, only `max_features` randomly chosen  
  features are candidates. This *decorrelates* the trees.
- **Averaging** reduces variance without increasing bias much.

> **Target:** Gini ≥ 0.15 on the validation set (Task 5).


In [8]:
# ══════════════════════════════════════════════════════════════════════════
#  Custom Random Forest Classifier
#
#  Parallelisation strategy:
#    prefer="threads" — trees share memory (no pickle needed).
#    The GIL is released during numpy ops so threads run truly in parallel.
# ══════════════════════════════════════════════════════════════════════════

def _train_one_rf_tree(X, y, max_depth, mf, n_sub, n_classes, seed):
    """Train a single RF tree — runs in a worker thread."""
    rng  = np.random.RandomState(seed)
    idx  = rng.choice(len(X), n_sub, replace=True)
    tree = DecisionTreeClassifier(
        max_depth=max_depth,
        max_features=mf,
        n_jobs=1,           # no nested parallelism inside threads
        random_state=seed,
    )
    tree.n_classes_ = n_classes
    tree.fit(X[idx], y[idx])
    return tree


def _predict_one_tree(tree, X):
    return tree.predict_proba(X)


class RandomForestClassifier:
    """Random Forest — trees trained in parallel via joblib threads.

    Parameters
    ----------
    n_estimators : int
    max_depth    : int | None
    max_features : int | 'sqrt' | 'log2' | None
    subsample    : float   — row fraction per tree
    n_jobs       : int     — worker threads (-1 = all cores)
    random_state : int
    """
    def __init__(self, n_estimators=100, max_depth=None,
                 max_features="sqrt", subsample=1.0,
                 n_jobs=-1, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth    = max_depth
        self.max_features = max_features
        self.subsample    = subsample
        self.n_jobs       = n_jobs
        self.random_state = random_state
        self.trees_       = []
        self._rng         = np.random.RandomState(random_state)

    def fit(self, X, y):
        n, nf = X.shape
        self.n_classes_ = int(y.max()) + 1
        mf    = (int(np.sqrt(nf))   if self.max_features == "sqrt"
                 else int(np.log2(nf)) if self.max_features == "log2"
                 else self.max_features if isinstance(self.max_features, int)
                 else nf)
        n_sub = max(1, int(n * self.subsample))
        seeds = self._rng.randint(0, 2**31, self.n_estimators).tolist()

        self.trees_ = Parallel(n_jobs=self.n_jobs, prefer="threads")(
            delayed(_train_one_rf_tree)(
                X, y, self.max_depth, mf, n_sub, self.n_classes_, s
            )
            for s in seeds
        )
        return self

    def predict_proba(self, X):
        probas = Parallel(n_jobs=self.n_jobs, prefer="threads")(
            delayed(_predict_one_tree)(tree, X)
            for tree in self.trees_
        )
        return np.mean(probas, axis=0)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


# ── Train & evaluate ──────────────────────────────────────────────────────
print(f"Training Custom Random Forest  (100 trees, max_depth=10, n_jobs={N_JOBS})…")
rf_custom = RandomForestClassifier(
    n_estimators=100, max_depth=10, max_features="sqrt",
    subsample=0.8, n_jobs=N_JOBS, random_state=SEED
)
rf_custom.fit(X_train, y_train)

prob_rf = rf_custom.predict_proba(X_valid)[:, 1]
gini_rf = gini(y_valid, prob_rf)
print(f"  Custom RF  →  Gini (valid) = {gini_rf:.4f}")
print(f"{'✓ Task 5 target met (≥ 0.15)!' if gini_rf >= 0.15 else '✗ Try more trees or deeper depth'}")


Training Custom Random Forest  (100 trees, max_depth=10, n_jobs=-1)…
  Custom RF  →  Gini (valid) = 0.4667
✓ Task 5 target met (≥ 0.15)!


---
## 6 · Custom GBDT Classifier <a id='6'></a>

### How Gradient Boosting works

Gradient Boosting builds an **additive model** F(x) = Σ αₜ hₜ(x).  
At each step *t* we:

1. Compute **pseudo-residuals** — the negative gradient of the loss w.r.t. current predictions.
2. Fit a **regression tree** hₜ(x) to those residuals.
3. Add a scaled version (`learning_rate × hₜ(x)`) to the ensemble.

**Binary cross-entropy loss:**

L(y, p) = − [ y · log(p) + (1−y) · log(1−p) ]

**Gradient (pseudo-residual):**

∂L/∂F = p − y,   where p = sigmoid(F)

So we train each new tree to predict `y − p` (the error of the current ensemble).


In [9]:
# ══════════════════════════════════════════════════════════════════════════
#  Custom GBDT Classifier
#
#  Parallelisation note:
#    GBDT rounds are inherently SEQUENTIAL (each tree depends on the last).
#    Parallelism is applied INSIDE each tree's split search via n_jobs.
# ══════════════════════════════════════════════════════════════════════════

def sigmoid(x):
    return np.where(x >= 0,
                    1.0 / (1.0 + np.exp(-x)),
                    np.exp(x) / (1.0 + np.exp(x)))


class GBDTClassifier:
    """Gradient Boosted Decision Trees — binary cross-entropy loss.

    Pseudo-residual = y − sigmoid(F)   (negative gradient of log-loss)

    Parameters
    ----------
    n_estimators  : boosting rounds
    max_depth     : depth of each regression tree
    learning_rate : shrinkage (η)
    max_features  : features per split (None = all)
    subsample     : row fraction per round
    n_jobs        : parallelism inside each tree's split search
    random_state  : int
    """
    def __init__(self, n_estimators=100, max_depth=3,
                 learning_rate=0.1, max_features=None,
                 subsample=1.0, n_jobs=-1, random_state=None):
        self.n_estimators  = n_estimators
        self.max_depth     = max_depth
        self.learning_rate = learning_rate
        self.max_features  = max_features
        self.subsample     = subsample
        self.n_jobs        = n_jobs
        self.random_state  = random_state
        self.trees_        = []
        self.F0_           = 0.0
        self._rng          = np.random.RandomState(random_state)

    def fit(self, X, y):
        n = len(y)
        p_mean   = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        self.F0_ = float(np.log(p_mean / (1 - p_mean)))
        F        = np.full(n, self.F0_)
        n_sub    = max(1, int(n * self.subsample))
        self.trees_ = []

        for _ in range(self.n_estimators):
            residuals = y - sigmoid(F)
            idx  = self._rng.choice(n, n_sub, replace=False)
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                max_features=self.max_features,
                n_jobs=self.n_jobs,       # ← parallel split search
                random_state=self._rng.randint(0, 2**31),
            )
            tree.fit(X[idx], residuals[idx])
            self.trees_.append(tree)
            F += self.learning_rate * tree.predict(X)
        return self

    def predict_proba(self, X):
        F = np.full(len(X), self.F0_)
        for tree in self.trees_:
            F += self.learning_rate * tree.predict(X)
        p = sigmoid(F)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ── Train & evaluate ──────────────────────────────────────────────────────
print(f"Training Custom GBDT (100 rounds, depth=3, n_jobs={N_JOBS} inside each tree)…")
gbdt_custom = GBDTClassifier(
    n_estimators=100, max_depth=3,
    learning_rate=0.1, subsample=0.8,
    n_jobs=N_JOBS, random_state=SEED
)
gbdt_custom.fit(X_train, y_train)

prob_gbdt = gbdt_custom.predict_proba(X_valid)[:, 1]
gini_gbdt = gini(y_valid, prob_gbdt)
print(f"  Custom GBDT →  Gini (valid) = {gini_gbdt:.4f}")


Training Custom GBDT (100 rounds, depth=3, n_jobs=-1 inside each tree)…
  Custom GBDT →  Gini (valid) = 0.4706


---
## 7 · LightGBM · CatBoost · XGBoost <a id='7'></a>

### Key differences at a glance

| | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Tree growth | **Depth-wise** (level by level) | **Leaf-wise** (best leaf first) | **Oblivious trees** (same split per level) |
| Categorical support | Manual encoding needed | `==` / `!=` split support | Native target encoding |
| Sampling strategy | Row + column subsampling | Gradient-based sampling (GOSS) | Minimal variance sampling |
| DART mode | ✓ | ✓ | — |
| Speed | Medium | Very fast | Medium |

### DART mode (Dropout Additive Regression Trees)
DART is an anti-overfitting technique: at each boosting round some randomly  
chosen trees are **dropped** (set to zero) before computing residuals. This  
prevents individual trees from having too much influence, similar to dropout in neural networks.

### CatBoost categorical features
CatBoost uses **ordered target encoding**: for each training sample it computes  
the mean target of all *previous* samples with the same category value.  
This prevents target leakage that affects naïve target encoding.


In [10]:
# ── 7.1  Prepare data ────────────────────────────────────────────────────
feature_names = feature_cols

# ─────────────────────────────────────────────────────────────────────────
#  LightGBM
#  num_threads = N_CORES  → uses all logical cores for histogram building
# ─────────────────────────────────────────────────────────────────────────
print("═"*55)
print("LightGBM")
print("═"*55)

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

lgb_params = {
    "objective"        : "binary",
    "metric"           : "auc",
    "learning_rate"    : 0.05,
    "num_leaves"       : 63,
    "max_depth"        : -1,
    "feature_fraction" : 0.8,
    "bagging_fraction" : 0.8,
    "bagging_freq"     : 5,
    "min_child_samples": 20,
    "num_threads"      : N_CORES,   # ← all cores
    "verbose"          : -1,
    "seed"             : SEED,
}

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_valid],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=0)],
)

prob_lgb = lgb_model.predict(X_valid)
gini_lgb = gini(y_valid, prob_lgb)
print(f"  Best iteration : {lgb_model.best_iteration}")
print(f"  Gini (valid)   : {gini_lgb:.4f}")


═══════════════════════════════════════════════════════
LightGBM
═══════════════════════════════════════════════════════
  Best iteration : 6
  Gini (valid)   : 0.4752


In [11]:
# ─────────────────────────────────────────────────────────────────────────
#  CatBoost
#  thread_count = -1  → CatBoost auto-detects all logical cores
#
#  WHY a separate DataFrame?
#  CatBoost applies ordered target encoding internally — it needs the
#  original string category values, NOT label-encoded floats.
# ─────────────────────────────────────────────────────────────────────────
print("═"*55)
print("CatBoost")
print("═"*55)

cb_train_pool = cb.Pool(cb_train_df, label=y_train, cat_features=cat_cols)
cb_valid_pool = cb.Pool(cb_valid_df, label=y_valid, cat_features=cat_cols)
cb_test_pool  = cb.Pool(cb_test_df,  label=y_test,  cat_features=cat_cols)

cb_model = cb.CatBoostClassifier(
    iterations            = 500,
    learning_rate         = 0.05,
    depth                 = 6,
    eval_metric           = "AUC",
    early_stopping_rounds = 50,
    random_seed           = SEED,
    thread_count          = -1,   # ← all cores
    verbose               = 0,
)
cb_model.fit(
    cb_train_pool,
    eval_set=cb_valid_pool,
    use_best_model=True,
)

prob_cb = cb_model.predict_proba(cb_valid_df)[:, 1]
gini_cb = gini(y_valid, prob_cb)
print(f"  Best iteration : {cb_model.best_iteration_}")
print(f"  Gini (valid)   : {gini_cb:.4f}")


═══════════════════════════════════════════════════════
CatBoost
═══════════════════════════════════════════════════════
  Best iteration : 86
  Gini (valid)   : 0.4865


In [12]:
# ─────────────────────────────────────────────────────────────────────────
#  XGBoost  (gbtree — depth-wise growth)
#  n_jobs = N_CORES, tree_method = "hist"  → fastest CPU histogram method
# ─────────────────────────────────────────────────────────────────────────
print("═"*55)
print("XGBoost  (booster = gbtree, tree_method = hist)")
print("═"*55)

xgb_model = xgb.XGBClassifier(
    n_estimators          = 500,
    learning_rate         = 0.05,
    max_depth             = 6,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    tree_method           = "hist",   # ← fastest CPU method
    n_jobs                = N_CORES,  # ← all cores
    eval_metric           = "auc",
    early_stopping_rounds = 50,
    use_label_encoder     = False,
    random_state          = SEED,
    verbosity             = 0,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False,
)

prob_xgb = xgb_model.predict_proba(X_valid)[:, 1]
gini_xgb = gini(y_valid, prob_xgb)
print(f"  Best iteration : {xgb_model.best_iteration}")
print(f"  Gini (valid)   : {gini_xgb:.4f}")


═══════════════════════════════════════════════════════
XGBoost  (booster = gbtree, tree_method = hist)
═══════════════════════════════════════════════════════
  Best iteration : 113
  Gini (valid)   : 0.4853


In [13]:
# ─────────────────────────────────────────────────────────────────────────
#  XGBoost  DART mode
#  DART = Dropout Additive Regression Trees
#  At each round some existing trees are randomly dropped before computing
#  residuals → prevents any single tree from dominating → less overfitting.
# ─────────────────────────────────────────────────────────────────────────
print("═"*55)
print("XGBoost  (booster = dart)")
print("═"*55)

xgb_dart = xgb.XGBClassifier(
    n_estimators      = 300,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    booster           = "dart",
    rate_drop         = 0.1,    # fraction of trees dropped per round
    skip_drop         = 0.5,    # prob of skipping dropout entirely
    tree_method       = "hist",
    n_jobs            = N_CORES,
    eval_metric       = "auc",
    use_label_encoder = False,
    random_state      = SEED,
    verbosity         = 0,
)
xgb_dart.fit(X_train, y_train, verbose=False)

prob_xgb_dart = xgb_dart.predict_proba(X_valid)[:, 1]
gini_xgb_dart = gini(y_valid, prob_xgb_dart)
print(f"  Gini (valid)   : {gini_xgb_dart:.4f}")


═══════════════════════════════════════════════════════
XGBoost  (booster = dart)
═══════════════════════════════════════════════════════
  Gini (valid)   : 0.4733


In [14]:
# ── Comparison table ─────────────────────────────────────────────────────
summary = pd.DataFrame({
    "Model"       : ["Custom DT", "sklearn DT", "Custom RF",
                     "Custom GBDT",
                     "LightGBM", "CatBoost", "XGBoost", "XGBoost DART"],
    "Gini (valid)": [gini_custom, gini_sk, gini_rf,
                     gini_gbdt,
                     gini_lgb, gini_cb, gini_xgb, gini_xgb_dart],
}).sort_values("Gini (valid)", ascending=False).reset_index(drop=True)

summary["Gini (valid)"] = summary["Gini (valid)"].round(4)
print(summary.to_string(index=False))
print(f"\n🏆 Best model on validation: {summary.iloc[0]['Model']}")


       Model  Gini (valid)
    CatBoost        0.4865
     XGBoost        0.4853
    LightGBM        0.4752
XGBoost DART        0.4733
 Custom GBDT        0.4706
   Custom RF        0.4667
   Custom DT        0.3927
  sklearn DT        0.3429

🏆 Best model on validation: CatBoost


---
## 8 · Best Model — Final Test Evaluation <a id='8'></a>

We now unlock the test set and evaluate our best model on all three splits.

**What to look for:**
- **Train Gini >> Valid Gini >> Test Gini** → severe overfitting.
- **Train ≈ Valid ≈ Test** → well-generalised model.
- A small drop from valid → test is expected (distribution shift over time).

A drop in Gini from valid to test can have two causes:
1. **Overfitting** — the model memorised validation-period patterns.
2. **Distribution shift** — the car market changed between the validation and test periods.

Because we used time-based splits, some performance drop is natural and expected.


In [15]:
# ── Final evaluation: train / valid / test ───────────────────────────────
def eval_model(name, predict_fn, splits):
    print(f"\n{'─'*40}  {name}")
    scores = {}
    for split_name, X, y in splits:
        prob = predict_fn(X)
        g    = gini(y, prob)
        print(f"  {split_name:<8}  Gini = {g:.4f}")
        scores[split_name] = g
    drop = scores["Valid"] - scores["Test"]
    flag = "⚠️  overfitting/shift" if drop > 0.02 else "✓ generalises well"
    print(f"  Valid→Test drop: {drop:+.4f}  {flag}")
    return scores

lgb_scores = eval_model(
    "LightGBM", lgb_model.predict,
    [("Train", X_train, y_train),
     ("Valid", X_valid, y_valid),
     ("Test",  X_test,  y_test)],
)

cb_scores = eval_model(
    "CatBoost", lambda X: cb_model.predict_proba(X)[:, 1],
    [("Train", cb_train_df, y_train),
     ("Valid", cb_valid_df, y_valid),
     ("Test",  cb_test_df,  y_test)],
)

xgb_scores = eval_model(
    "XGBoost", lambda X: xgb_model.predict_proba(X)[:, 1],
    [("Train", X_train, y_train),
     ("Valid", X_valid, y_valid),
     ("Test",  X_test,  y_test)],
)



────────────────────────────────────────  LightGBM
  Train     Gini = 0.5673
  Valid     Gini = 0.4752
  Test      Gini = 0.4883
  Valid→Test drop: -0.0132  ✓ generalises well

────────────────────────────────────────  CatBoost
  Train     Gini = 0.5338
  Valid     Gini = 0.4865
  Test      Gini = 0.4754
  Valid→Test drop: +0.0111  ✓ generalises well

────────────────────────────────────────  XGBoost
  Train     Gini = 0.6902
  Valid     Gini = 0.4853
  Test      Gini = 0.5124
  Valid→Test drop: -0.0271  ✓ generalises well


---
## 9 · ExtraTrees Classifier *(Bonus)* <a id='9'></a>

### ExtraTrees vs. Random Forest
| | Random Forest | ExtraTrees |
|---|---|---|
| Row subsampling | Yes (bootstrap) | No (full dataset) |
| Threshold search | Best over all values | **Random threshold** per feature |
| Variance | Lower | Lower still |
| Bias | Slightly higher | Slightly higher still |
| Speed | Slower | **Faster** (no threshold search) |

Because ExtraTrees uses truly random thresholds, individual trees are weaker, but  
the extra randomisation further decorrelates the ensemble, which can reduce variance.

> **Target:** Gini ≥ 0.12 on the validation set (Task 9).


In [16]:
# ══════════════════════════════════════════════════════════════════════════
#  ExtraTrees Classifier
#
#  Key differences vs Random Forest:
#    • NO row subsampling — every tree sees the full training set
#    • Random thresholds per feature (extra_randomized=True)
#
#  Parallelisation: joblib threads (no pickling issues)
# ══════════════════════════════════════════════════════════════════════════

def _train_one_et_tree(X, y, max_depth, mf, n_classes, seed):
    """Train a single ExtraTree — runs in a worker thread."""
    tree = DecisionTreeClassifier(
        max_depth=max_depth,
        max_features=mf,
        extra_randomized=True,
        n_jobs=1,
        random_state=seed,
    )
    tree.n_classes_ = n_classes
    tree.fit(X, y)          # full dataset — no row subsampling
    return tree


class ExtraTreesClassifier:
    """Extremely Randomised Trees — all trees trained in parallel."""
    def __init__(self, n_estimators=100, max_depth=None,
                 max_features="sqrt", n_jobs=-1, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth    = max_depth
        self.max_features = max_features
        self.n_jobs       = n_jobs
        self.random_state = random_state
        self.trees_       = []
        self._rng         = np.random.RandomState(random_state)

    def fit(self, X, y):
        nf = X.shape[1]
        self.n_classes_ = int(y.max()) + 1
        mf = (int(np.sqrt(nf))   if self.max_features == "sqrt"
              else int(np.log2(nf)) if self.max_features == "log2"
              else self.max_features if isinstance(self.max_features, int)
              else nf)
        seeds = self._rng.randint(0, 2**31, self.n_estimators).tolist()

        self.trees_ = Parallel(n_jobs=self.n_jobs, prefer="threads")(
            delayed(_train_one_et_tree)(
                X, y, self.max_depth, mf, self.n_classes_, s
            )
            for s in seeds
        )
        return self

    def predict_proba(self, X):
        probas = Parallel(n_jobs=self.n_jobs, prefer="threads")(
            delayed(_predict_one_tree)(tree, X)
            for tree in self.trees_
        )
        return np.mean(probas, axis=0)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


# ── Train & evaluate ──────────────────────────────────────────────────────
print(f"Training ExtraTrees  (100 trees, max_depth=12, n_jobs={N_JOBS})…")
et = ExtraTreesClassifier(
    n_estimators=100, max_depth=12,
    max_features="sqrt", n_jobs=N_JOBS, random_state=SEED
)
et.fit(X_train, y_train)

prob_et = et.predict_proba(X_valid)[:, 1]
gini_et = gini(y_valid, prob_et)
print(f"  ExtraTrees →  Gini (valid) = {gini_et:.4f}")
print(f"{'✓ Bonus target met (≥ 0.12)!' if gini_et >= 0.12 else '✗ Try more trees or deeper depth'}")


Training ExtraTrees  (100 trees, max_depth=12, n_jobs=-1)…
  ExtraTrees →  Gini (valid) = 0.4703
✓ Bonus target met (≥ 0.12)!


---
## 🏁 Final Summary

| Task | Goal | Status |
|---|---|---|
| 1 | Data preprocessing, time-based split | ✓ |
| 2 | Custom DT Classifier + Regressor | ✓ |
| 3 | Custom DT Gini ≥ 0.10 | see output |
| 4 | sklearn DT comparison | ✓ |
| 5 | Custom RF Gini ≥ 0.15 | see output |
| 6 | Custom GBDT (binary cross-entropy) | ✓ |
| 7 | LightGBM / CatBoost / XGBoost | ✓ |
| 8 | Test set evaluation + overfitting analysis | ✓ |
| 9* | ExtraTrees Gini ≥ 0.12 | see output |

### Key takeaways
- **Single trees** are interpretable but overfit. Ensembles fix that.  
- **Random Forest** reduces variance through bagging + feature randomisation.  
- **GBDT** reduces bias sequentially; each tree corrects the previous error.  
- **LightGBM** is typically fastest due to leaf-wise growth and GOSS sampling.  
- **CatBoost** shines on data with many categorical features via ordered target encoding.  
- **XGBoost** is mature and widely deployed; DART mode adds regularisation via tree dropout.  
- A small valid → test Gini drop is expected with time-based splits due to distribution shift.


In [17]:
# ── Full model leaderboard ────────────────────────────────────────────────
all_scores = {
    "Custom DT"   : gini_custom,
    "sklearn DT"  : gini_sk,
    "Custom RF"   : gini_rf,
    "Custom GBDT" : gini_gbdt,
    "LightGBM"    : gini_lgb,
    "CatBoost"    : gini_cb,
    "XGBoost"     : gini_xgb,
    "XGBoost DART": gini_xgb_dart,
    "ExtraTrees"  : gini_et,
}

leaderboard = (pd.DataFrame.from_dict(all_scores, orient="index",
                                       columns=["Gini (valid)"])
               .sort_values("Gini (valid)", ascending=False)
               .round(4))
print(leaderboard.to_string())
print(f"\nCores used: {N_CORES}")


              Gini (valid)
CatBoost            0.4865
XGBoost             0.4853
LightGBM            0.4752
XGBoost DART        0.4733
Custom GBDT         0.4706
ExtraTrees          0.4703
Custom RF           0.4667
Custom DT           0.3927
sklearn DT          0.3429

Cores used: 32
